In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, precision_score, f1_score
from imblearn.over_sampling import SMOTE  
from imblearn.pipeline import Pipeline     
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import  RandomizedSearchCV,GridSearchCV
from sklearn.metrics import classification_report,accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

warnings.filterwarnings(action='ignore')
%config InlineBackend.figure_format = 'retina' ##


In [2]:
path = '../../dataset/preprocessed/hotel_bookings_dummy.csv'
df = pd.read_csv(path)

In [3]:
df.head()

,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,diff_reserved_room_type,is_agent,is_company,is_city_hotel,arrival_date_month_August,arrival_date_month_December,arrival_date_month_February,arrival_date_month_January,arrival_date_month_July,arrival_date_month_June,arrival_date_month_March,arrival_date_month_May,arrival_date_month_November,arrival_date_month_October,arrival_date_month_September,meal_FB,meal_HB,meal_SC,meal_Undefined,market_segment_Complementary,market_segment_Corporate,market_segment_Direct,market_segment_Groups,market_segment_Offline TA/TO,market_segment_Online TA,market_segment_Undefined,distribution_channel_Direct,distribution_channel_GDS,distribution_channel_TA/TO,distribution_channel_Undefined,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Group,customer_type_Transient,customer_type_Transient-Party,guest_type_Group,guest_type_Single,guest_type_Undefined,country_region_OTH,country_region_PRT
0,0,342,0,0,0,0,0,3,0,0.0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,1
1,0,737,0,0,0,0,0,4,0,0.0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,1
2,0,7,0,1,0,0,0,0,0,75.0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0
3,0,13,0,1,0,0,0,0,0,75.0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0
4,0,14,0,2,0,0,0,0,0,98.0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0


In [4]:
X = df.drop(['is_canceled'],axis =1)
y = df['is_canceled']
X = pd.get_dummies(X, drop_first=True,dtype = float)


In [5]:
# 데이터 분리
X_train,X_test,y_train, y_test = train_test_split(X,y,
                                                  train_size=0.7,
                                                  random_state=1004)

In [6]:
#=============== 모델 생성 및 학습 ================
model_xgb = XGBClassifier()

model_xgb.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [7]:
#=================모델 예측 및 결과확인 ====================
y_pred =model_xgb.predict(X_test)

cv_score = cross_val_score(model_xgb,X_train,y_train, cv = 10, scoring = 'f1')
print(cv_score.mean())
print(cv_score.std())

0.814568489917287
0.0030154849940027244


In [8]:
# 진짜 최종 f1-score 계산 및 출력
final_f1 = f1_score(y_test, y_pred)
print(f"최종 Test Set f1-score: {final_f1}")

최종 Test Set f1-score: 0.8152566316119918


In [9]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.92      0.90     22390
           1       0.85      0.78      0.82     13281

    accuracy                           0.87     35671
   macro avg       0.86      0.85      0.86     35671
weighted avg       0.87      0.87      0.87     35671



In [10]:
print('F1-Score:', f1_score(y_test, y_pred, average=None))

F1-Score: [0.89733525 0.81525663]


In [11]:
test_f1 = f1_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)

print(f"  Recall   : {test_recall:.3f}")
print(f"  Precision  : {test_precision:.3f}")
print(f"  F1-Score     : {test_f1:.3f}")


  Recall   : 0.782
  Precision  : 0.851
  F1-Score     : 0.815


In [12]:
param_dist_xgb = {
    'n_estimators': [50, 100, 150, 200],         # 1. 나무의 총 개수
    'max_depth': [3, 5, 7, 9, 12],                # 2. 나무의 깊이 (XGBoost는 3~10 사이가 적당)
    'learning_rate': [0.01, 0.05, 0.1, 0.2],      # 3. 학습률 (이전 나무의 오차를 얼마나 강하게 반영할 것인가)
    'min_child_weight': [1, 3, 5, 7],             # 4. 랜덤 포레스트의 min_samples_leaf와 유사 (과적합 방지용)
    'subsample': [0.6, 0.8, 1.0],                 # 5. 나무를 만들 때 사용할 행(데이터)의 비율 (과적합 방지)
    'colsample_bytree': [0.6, 0.8, 1.0]           # 6. 랜덤 포레스트의 max_features와 유사 (사용할 열의 비율)
}

# 3. 교차 검증 조건 설정 (동일하게 5-Fold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1004)

# 4. XGBoost 전용 RandomizedSearchCV 기계 가동!
random_search_xgb = RandomizedSearchCV(
    estimator=model_xgb,           
    param_distributions=param_dist_xgb, # XGBoost 전용 후보군
    n_iter=15,                          # 속도가 빠르니 10개 대신 15개 조합을 봐도 금방 끝납니다!
    scoring='f1',                       # 채점 기준은 언제나 든든한 F1-Score
    cv=cv,
    n_jobs=-1,                    
    random_state=1004 
)

In [13]:
random_search_xgb.fit(X_train,y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.6, 0.8, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 5, ...], 'min_child_weight': [1, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used he

In [14]:
print(random_search_xgb.cv_results_['mean_test_score'])

print('최적의 파라미터: ',random_search_xgb.best_params_)

print('최적의 성능:',random_search_xgb.best_score_)

[0.81291299 0.57659917 0.8013618  0.77796136 0.58053737 0.78263768
 0.48587918 0.81847739 0.81532606 0.79757143 0.81508558 0.77545679
 0.52956332 0.57611163 0.77776414]
최적의 파라미터:  {'subsample': 1.0, 'n_estimators': 200, 'min_child_weight': 7, 'max_depth': 7, 'learning_rate': 0.2, 'colsample_bytree': 0.6}
최적의 성능: 0.8184773948475783


In [ ]:
final_model_xgb = random_search_xgb.best_estimator_

y_pred_xgb = final_model_xgb.predict(X_test)

print(f"랜덤 서치 모델의 f1-score {f1_score(y_test, y_pred_xgb)}")

최종 모델의 f1-score 0.817483339866719


In [20]:
param_grid_xgb = {
    'n_estimators': [100, 150, 200],              # 심을 나무의 개수
    'max_depth': [5, 7, 9, 11],                   # 트리 최대 깊이 (XGBoost는 보통 3~10 사이가 핵심)
    'learning_rate': [0.05, 0.1, 0.2, 0.3],        # 학습률 (디폴트 0.3 주변 정밀 타격)
    'subsample': [0.8, 1.0]                        # 각 나무를 심을 때 사용할 데이터 샘플 비율
}

# 3. 교차 검증 조건 고정 (동일하게 5-Fold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1004)

# 4. GridSearchCV 기계 가동 (모든 조합 전수조사!)
grid_search_xgb = GridSearchCV(
    estimator=model_xgb,           
    param_grid=param_grid_xgb,      # XGBoost용 그물망 딕셔너리 적용
    scoring='f1',                  # 채점 기준 동일하게 F1-Score
    cv=cv,
    n_jobs=-1                      # CPU 풀가동
)

In [21]:
grid_search_xgb.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.05, 0.1, ...], 'max_depth': [5, 7, ...], 'n_estimators': [100, 150, ...], 'subsample': [0.8, 1.0]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time 

In [22]:
print(grid_search_xgb.cv_results_['mean_test_score'])

print('최적의 파라미터: ',grid_search_xgb.best_params_)

print('최적의 성능:',grid_search_xgb.best_score_)

[0.77839891 0.77725911 0.78658444 0.78460519 0.79190361 0.79083745
 0.78745322 0.78700228 0.79700322 0.7961455  0.80279709 0.80135138
 0.79831986 0.79555505 0.80800369 0.80382034 0.81478709 0.81191206
 0.80853185 0.80556256 0.81638627 0.81301448 0.82146383 0.81850618
 0.79174688 0.79108316 0.79804276 0.79744329 0.80300448 0.80201519
 0.8023726  0.80156329 0.81179097 0.80980006 0.81689813 0.81459447
 0.81346832 0.81123681 0.81965242 0.81805013 0.82323551 0.82188305
 0.82084723 0.81899331 0.82651892 0.82326049 0.82876619 0.82541162
 0.80289021 0.80211873 0.81012441 0.80835149 0.8131324  0.81100474
 0.81532606 0.8135785  0.81970489 0.81882002 0.82105628 0.82144136
 0.8220246  0.82088236 0.82299771 0.82494813 0.82427808 0.82640371
 0.82545973 0.82529063 0.82656357 0.82676378 0.82676829 0.82678686
 0.80814713 0.80731409 0.81293409 0.81288896 0.81645544 0.81486431
 0.81644558 0.81730959 0.81937179 0.81996491 0.82040415 0.82255527
 0.8221923  0.8241685  0.82363683 0.82551636 0.82228349 0.8270

In [23]:
#gridsearch 후 최고 모델 선택
final_model_xgb_2 = grid_search_xgb.best_estimator_

y_pred_xgb2 = final_model_xgb_2.predict(X_test)

print(f"그리드 서치 최종 모델의 f1-score {f1_score(y_test,y_pred_xgb2)}")

그리드 서치 최종 모델의 f1-score 0.8284281326542514


In [26]:
# 두 모델의 '취소 F1-Score'를 먼저 계산해서 점수 비교하기
score_random = f1_score(y_test, y_pred_xgb)
score_grid = f1_score(y_test, y_pred_xgb2)

print(f"랜덤 서치 F1-score: {score_random:.4f} | 그리드 서치 F1: {score_grid:.4f}")

# 2. 유동적 모델 선택 
if score_grid >= score_random:
    print("그리드 서치 모델 선택")
    final_preds = y_pred_xgb2
    model_name = "XGBoost (Grid)"
else:
    print("랜덤 서치 모델 선택")
    final_preds = y_pred_xgb
    model_name = "XGBoost (Random)"

# 최종 선택된 승자 모델의 예측값(final_preds)으로 핵심 지표 계산
final_accuracy = accuracy_score(y_test, final_preds)

# 클래스별 Recall 중 1번(취소)만 추출
final_each_recall = recall_score(y_test, final_preds, average=None)
final_cancel_recall = final_each_recall[1]

# 클래스별 F1 중 1번(취소)만 추출
final_each_f1 = f1_score(y_test, final_preds, average=None)
final_cancel_f1 = final_each_f1[1]

# 결과 데이터프레임 생성 
xgb_res = pd.DataFrame({
    'Model': ['XGBoost'],          
    'Accuracy': [final_accuracy],        
    'Recall': [final_cancel_recall],   
    'F1_Score': [final_cancel_f1]      
})

# 개별 값 출력해서 확인
print()
print(f"최종 저장될 Accuracy : {final_accuracy:.4f}")
print(f"최종 저장될 Recall   : {final_cancel_recall:.4f}")
print(f"최종 저장될 F1_Score : {final_cancel_f1:.4f}")


# 'visualization/model_results' 폴더로 연결 및 저장
output_dir = '../../visualization/model_results'
os.makedirs(output_dir, exist_ok=True) 

# csv 파일로 최종 저장 
xgb_res.to_csv(f'{output_dir}/xgb_result.csv', index=False)
print(f"\n전송완료 {output_dir}/xgb_result.csv")

랜덤 서치 F1-score: 0.8175 | 그리드 서치 F1: 0.8284
그리드 서치 모델 선택

최종 저장될 Accuracy : 0.8763
최종 저장될 Recall   : 0.8022
최종 저장될 F1_Score : 0.8284

전송완료 ../../visualization/model_results/xgb_result.csv
